# Load needed modules

In [1]:
# modules:
import pandas as pd
import os
from openai import OpenAI



from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.callbacks import get_openai_callback


import json


# Load data to translate
> this should be the drawn concept data set whereby 1 row is 1 drawn concept

In [2]:
print(os.getcwd())
# Get the current working directory
os.chdir("outputs/associations_translated") 
directory = os.getcwd()

/home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses


In [3]:
# print the current working directory
print(directory)

/home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses/outputs/associations_translated


In [4]:
# List files in the current working directory
files = os.listdir('.')
# Display the list of files
print(files)

['ass_study_translated.csv', '.~lock.ass_study_translated_man.xlsx#', 'ass_study_translated_man.xlsx', 'ass_study_translated.xlsx']


In [12]:
sheet1 = pd.read_excel(f"{directory}/ass_study_translated_man.xlsx", sheet_name=0)
sheet2 = pd.read_excel(f"{directory}/ass_study_translated_man.xlsx", sheet_name=1)

print("sheet1:", sheet1.shape)
print("sheet2:", sheet2.shape)

sheet1: (18960, 12)
sheet2: (465, 12)


In [13]:
sheet1

,participant_id,study_condition,gender,age,cue,valence,response,response_position,response_level,timestamp,time_diff_sec,response_translated
0,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,6.0,Richtig,1,1,2025-11-24 08:16:27.162,0.000,correct
1,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,7.0,Fortschritt,2,1,2025-11-24 08:16:35.087,7.925,progress
2,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,5.0,Zukunft,3,1,2025-11-24 08:16:38.866,11.704,future
3,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,6.0,Zusammenspiel,4,1,2025-11-24 08:16:46.704,19.542,interaction
4,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,7.0,Verbesserung,5,1,2025-11-24 08:16:54.584,27.422,improvement
...,...,...,...,...,...,...,...,...,...,...,...,...
18955,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,wieso,1,2,2026-02-11 17:13:58.343,443.097,NaN
18956,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,künstlich,2,2,2026-02-11 17:14:07.421,452.175,artificial
18957,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,unsicher,3,2,2026-02-11 17:14:15.217,459.971,NaN
18958,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,schlecht,4,2,2026-02-11 17:14:22.131,466.885,bad


In [14]:
sheet2

,participant_id,study_condition,gender,age,cue,valence,response,response_position,response_level,timestamp,time_diff_sec,response_translated
0,672dc833e51f3daa46245836,Diagnostic,male,35,Zukunft,NaN,"Wird ""normal"" werden",1,2,2025-11-24 08:32:07.597,758.516,"will become ""normal"""
1,69156afae8918e584cb58ac2,PersonnelSelection,male,37,fehlerhaft,NaN,leicht auszutrickesen,5,2,2025-11-24 10:27:27.931,190.764,easy to trick
2,6914ede06fc0e510d6cbba1b,Diagnostic,male,31,akzeptabel,NaN,kritiker schnell überzeugbar,4,2,2025-11-24 10:38:18.510,397.235,critics easily convinced
3,610700f4f9547087ee11e1ca,Migration,female,29,Migration,7.0,Sinnvoll,1,1,2025-11-24 10:55:51.844,0.000,sensible
4,610700f4f9547087ee11e1ca,Migration,female,29,Migration,2.0,Bias,4,1,2025-11-24 10:56:05.771,13.927,bias
...,...,...,...,...,...,...,...,...,...,...,...,...
460,69273d7a43eca09e260bac35,CourtDecision,female,22,CourtDecision,NaN,Unsicher,1,1,2026-02-11 17:06:35.246,0.000,uncertain
461,69273d7a43eca09e260bac35,CourtDecision,female,22,Unsicher,NaN,Aufmerksam,1,2,2026-02-11 17:08:33.136,117.890,attentive
462,69273d7a43eca09e260bac35,CourtDecision,female,22,Unsicher,NaN,vertrauenswürdig,3,2,2026-02-11 17:08:48.426,133.180,trustworthy
463,69273d7a43eca09e260bac35,CourtDecision,female,22,Fragwürdig,NaN,wieso,1,2,2026-02-11 17:13:58.343,443.097,why


# Merge sheet1 and sheet2

In [15]:
# Merge response_translated from sheet2 into sheet1 based on matching key columns
# Key columns: participant_id, response_position, response_level, timestamp

# First, let's check what columns are available in both sheets
print("Sheet1 columns:", sheet1.columns.tolist())
print("Sheet2 columns:", sheet2.columns.tolist())
print()

# Check if the key columns exist in both sheets
key_columns = ['participant_id', 'response_position', 'response_level', 'timestamp']
missing_in_sheet1 = [col for col in key_columns if col not in sheet1.columns]
missing_in_sheet2 = [col for col in key_columns if col not in sheet2.columns]

if missing_in_sheet1:
    print(f"Warning: Missing key columns in sheet1: {missing_in_sheet1}")
if missing_in_sheet2:
    print(f"Warning: Missing key columns in sheet2: {missing_in_sheet2}")

# Check if response_translated exists in sheet2
if 'response_translated' not in sheet2.columns:
    print("Warning: 'response_translated' column not found in sheet2")
else:
    print("All required columns found. Proceeding with merge...")
    
    # Create a copy of sheet1 to avoid modifying the original
    sheet1_updated = sheet1.copy()
    
    # Method 1: Using pandas merge (recommended)
    # This will add response_translated from sheet2 to sheet1 where key columns match
    merged_df = sheet1_updated.merge(
        sheet2[key_columns + ['response_translated']], 
        on=key_columns, 
        how='left', 
        suffixes=('', '_from_sheet2')
    )
    
    # If sheet1 already has response_translated, we can choose to update it
    if 'response_translated' in sheet1_updated.columns:
        # Update existing response_translated with values from sheet2 where available
        merged_df['response_translated'] = merged_df['response_translated_from_sheet2'].fillna(merged_df['response_translated'])
        # Drop the temporary column
        if 'response_translated_from_sheet2' in merged_df.columns:
            merged_df = merged_df.drop('response_translated_from_sheet2', axis=1)
    else:
        # If no existing response_translated, just rename the column
        if 'response_translated_from_sheet2' in merged_df.columns:
            merged_df = merged_df.rename(columns={'response_translated_from_sheet2': 'response_translated'})
    
    # Report the results
    original_shape = sheet1.shape
    merged_shape = merged_df.shape
    
    print(f"Original sheet1 shape: {original_shape}")
    print(f"Merged dataframe shape: {merged_shape}")
    
    # Count successful matches
    if 'response_translated' in merged_df.columns:
        successful_matches = merged_df['response_translated'].notna().sum()
        print(f"Successfully matched and filled response_translated for {successful_matches} rows")
    
    # Store the result
    sheet1_final = merged_df
    
    print("\nMerge completed successfully!")

Sheet1 columns: ['participant_id', 'study_condition', 'gender', 'age', 'cue', 'valence', 'response', 'response_position', 'response_level', 'timestamp', 'time_diff_sec', 'response_translated']
Sheet2 columns: ['participant_id', 'study_condition', 'gender', 'age', 'cue', 'valence', 'response', 'response_position', 'response_level', 'timestamp', 'time_diff_sec', 'response_translated']

All required columns found. Proceeding with merge...
Original sheet1 shape: (18960, 12)
Merged dataframe shape: (18960, 12)
Successfully matched and filled response_translated for 18960 rows

Merge completed successfully!


In [16]:
# Verify the merge results
if 'sheet1_final' in locals():
    print("=== MERGE VERIFICATION ===")
    print(f"Original sheet1 rows: {len(sheet1)}")
    print(f"Final merged rows: {len(sheet1_final)}")
    print(f"Sheet2 rows with response_translated: {len(sheet2[sheet2['response_translated'].notna()])}")
    
    # Check for any duplicates created during merge
    duplicates = sheet1_final.duplicated(subset=key_columns).sum()
    print(f"Duplicate rows after merge: {duplicates}")
    
    # Count non-null response_translated values
    if 'response_translated' in sheet1_final.columns:
        non_null_translations = sheet1_final['response_translated'].notna().sum()
        print(f"Rows with response_translated: {non_null_translations}")
        
        # Show translation coverage
        total_responses = len(sheet1_final[sheet1_final['response'].notna()])
        coverage_percent = (non_null_translations / total_responses * 100) if total_responses > 0 else 0
        print(f"Translation coverage: {coverage_percent:.1f}% ({non_null_translations}/{total_responses})")
else:
    print("Error: sheet1_final not created. Please check the merge process above.")

=== MERGE VERIFICATION ===
Original sheet1 rows: 18960
Final merged rows: 18960
Sheet2 rows with response_translated: 465
Duplicate rows after merge: 0
Rows with response_translated: 18960
Translation coverage: 100.0% (18960/18960)


In [17]:
cols = ["response", "response_translated"]
missing_counts = sheet1_final[cols].isna().sum()
missing_pct = sheet1_final[cols].isna().mean() * 100

print("Missing counts and percentages (of rows):")
for c in cols:
    print(f"{c}: {missing_counts[c]} / {len(sheet1_final)} ({missing_pct[c]:.2f}%)")

# Show up to 10 examples of rows with missing values for each column in `cols`
for c in cols:
    miss = sheet1_final[sheet1_final[c].isna()]
    n = len(miss)
    print(f"\nColumn '{c}' — missing {n}/{len(sheet1_final)} rows ({missing_pct[c]:.2f}%)")
    if n == 0:
        print("  (none)")
    else:
        # show context columns and up to 10 examples
        print(miss.loc[:, ['participant_id', 'cue', 'response', 'response_translated']].head(10).to_string(index=True))

Missing counts and percentages (of rows):
response: 0 / 18960 (0.00%)
response_translated: 0 / 18960 (0.00%)

Column 'response' — missing 0/18960 rows (0.00%)
  (none)

Column 'response_translated' — missing 0/18960 rows (0.00%)
  (none)


# test sentiment analysis (locally)

In [20]:
from transformers import pipeline
sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")

texts = [
    "I love this!",
    "I hate this!",
    "Sometimes I love this, sometimes I hate this!"
]

results = sentiment_analysis(texts)  # returns list of {'label':..., 'score':...}

def signed_confidence(res):
    return res['score'] if res['label'] == 'POSITIVE' else -res['score']

def centered_confidence(res):
    val = 2 * res['score'] - 1
    return val if res['label'] == 'POSITIVE' else -val

for t, r in zip(texts, results):
    print(t)
    print("raw:", r)
    print("signed_confidence:", signed_confidence(r))
    print("centered_confidence:", centered_confidence(r))
    print()

Device set to use cpu


I love this!
raw: {'label': 'POSITIVE', 'score': 0.9988656044006348}
signed_confidence: 0.9988656044006348
centered_confidence: 0.9977312088012695

I hate this!
raw: {'label': 'NEGATIVE', 'score': 0.9994561076164246}
signed_confidence: -0.9994561076164246
centered_confidence: -0.9989122152328491

Sometimes I love this, sometimes I hate this!
raw: {'label': 'NEGATIVE', 'score': 0.9672607779502869}
signed_confidence: -0.9672607779502869
centered_confidence: -0.9345215559005737



# apply sentiment analysis (locally)

In [21]:
# Apply sentiment analysis to response_translated column
from transformers import pipeline

# Initialize the sentiment analysis pipeline
sentiment_analysis = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english")

# Apply sentiment analysis to response_translated column
if 'response_translated' in sheet1_final.columns:
    # Get non-null response_translated values
    valid_responses = sheet1_final['response_translated'].dropna().astype(str)
    
    if len(valid_responses) > 0:
        print(f"Analyzing sentiment for {len(valid_responses)} translated responses...")
        
        # Perform sentiment analysis
        sentiment_results = sentiment_analysis(valid_responses.tolist())
        
        # Initialize new columns
        sheet1_final['sentiment_label'] = None
        sheet1_final['sentiment_score'] = None
        
        # Map results back to the dataframe
        for idx, result in zip(valid_responses.index, sentiment_results):
            sheet1_final.at[idx, 'sentiment_label'] = result['label']
            sheet1_final.at[idx, 'sentiment_score'] = signed_confidence(result)
        
        # Report results
        sentiment_counts = sheet1_final['sentiment_label'].value_counts()
        print(f"Sentiment analysis completed:")
        print(f"- POSITIVE: {sentiment_counts.get('POSITIVE', 0)}")
        print(f"- NEGATIVE: {sentiment_counts.get('NEGATIVE', 0)}")
        
        # Show statistics of sentiment scores
        scores = sheet1_final['sentiment_score'].dropna()
        if len(scores) > 0:
            print(f"\nSentiment score statistics:")
            print(f"- Mean: {scores.mean():.3f}")
            print(f"- Std: {scores.std():.3f}")
            print(f"- Min: {scores.min():.3f}")
            print(f"- Max: {scores.max():.3f}")
        
    else:
        print("No valid response_translated values found for sentiment analysis.")
        sheet1_final['sentiment_label'] = None
        sheet1_final['sentiment_score'] = None
else:
    print("response_translated column not found in sheet1_final.")

Device set to use cpu


Analyzing sentiment for 18960 translated responses...
Sentiment analysis completed:
- POSITIVE: 8832
- NEGATIVE: 10128

Sentiment score statistics:
- Mean: -0.057
- Std: 0.970
- Min: -1.000
- Max: 0.999
Sentiment analysis completed:
- POSITIVE: 8832
- NEGATIVE: 10128

Sentiment score statistics:
- Mean: -0.057
- Std: 0.970
- Min: -1.000
- Max: 0.999


In [22]:
cols = ["sentiment_label", "sentiment_score"]
missing_counts = sheet1_final[cols].isna().sum()
missing_pct = sheet1_final[cols].isna().mean() * 100

print("Missing counts and percentages (of rows):")
for c in cols:
    print(f"{c}: {missing_counts[c]} / {len(sheet1_final)} ({missing_pct[c]:.2f}%)")

Missing counts and percentages (of rows):
sentiment_label: 0 / 18960 (0.00%)
sentiment_score: 0 / 18960 (0.00%)


# save final dataset

In [23]:
# Save df_translated to current working directory as XLSX and CSV
filename_base = "ass_study_translated_sentiment"
xlsx_path = os.path.join(os.getcwd(), f"{filename_base}.xlsx")
csv_path = os.path.join(os.getcwd(), f"{filename_base}.csv")

try:
    sheet1_final.to_excel(xlsx_path, index=False)
    print(f"Saved Excel: {xlsx_path} ({sheet1_final.shape[0]} rows, {sheet1_final.shape[1]} cols)")
except Exception as e:
    print(f"Error saving Excel: {e}")

try:
    sheet1_final.to_csv(csv_path, index=False)
    print(f"Saved CSV:   {csv_path} ({sheet1_final.shape[0]} rows, {sheet1_final.shape[1]} cols)")
except Exception as e:
    print(f"Error saving CSV: {e}")

Saved Excel: /home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses/outputs/associations_translated/ass_study_translated_sentiment.xlsx (18960 rows, 14 cols)
Saved CSV:   /home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses/outputs/associations_translated/ass_study_translated_sentiment.csv (18960 rows, 14 cols)
